# MAI201 MLOps: Assignment 2: Data Validation & Testing

**Student:** Arushi Anand  



## Part 1: Great Expectations Setup

In [112]:
# 1.1  Verify installation and imports
import os, re, glob, shutil, subprocess, sys, warnings
import pandas as pd
import great_expectations as gx
import great_expectations.expectations as gxe

warnings.filterwarnings("ignore")

print(f"Python            : {sys.version.split()[0]}")
print(f"Great Expectations: {gx.__version__}")
print(f"Pandas            : {pd.__version__}")

Python            : 3.13.11
Great Expectations: 1.18.2
Pandas            : 3.0.3


In [113]:
# 1.2  Load dataset
DATA_FILE = r"C:\Users\suren\Documents\aru\MAI\Sem 2\ML Ops\Deliverables\Assignment 2_MLOps\customer_data.xlsx"

df_raw = pd.read_excel(DATA_FILE)
print(f"Shape  : {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(10)

Shape  : (5015, 7)
Columns: ['customer_id', 'age', 'email', 'salary', 'country', 'phone', 'signup_date']


,customer_id,age,email,salary,country,phone,signup_date
0,C04875,999.0,user4875@example.com,41017.0,USA,3637929158,2023-09-22 00:00:00
1,C01026,70.0,user1026@example.com,-1000.0,Australia,-8437,2023-02-25 00:00:00
2,C03546,79.0,user3546@example.com,82797.0,USA,423.366.4508,2023-09-15 00:00:00
3,C03822,20.0,@domain.com,54624.0,India,719-808-4765,2023-08-06 00:00:00
4,C04395,68.0,user4395@example.com,155942.0,USA,NaN,2023-05-28 00:00:00
5,C00923,70.0,user3423@example.com,99521.0,USA,970.521.2218,2023-10-07 00:00:00
6,C01991,77.0,user2418@example.com,59709.0,UK,(318) 414-9221,2023-07-29 00:00:00
7,C00380,74.0,user1096@example.com,NaN,UK,6945728232,2023-10-02 00:00:00
8,C04460,-25.0,user4460@example.com,49252.0,USA,502-394-4542,2023-04-14 00:00:00
9,C00706,61.0,user706@example.com,142935.0,Canada,553-249-1162,2023-05-01 00:00:00


In [114]:
# 1.3  Initialise a file-backed Great Expectations project
GE_ROOT = "gx_project"

if os.path.exists(GE_ROOT):
    shutil.rmtree(GE_ROOT)

context = gx.get_context(mode="file", project_root_dir=GE_ROOT)
print(f" GE project initialised at: {GE_ROOT}/")

 GE project initialised at: gx_project/


In [115]:
# 1.4  Configure Pandas data source
datasource       = context.data_sources.add_pandas("pandas_datasource")
data_asset       = datasource.add_dataframe_asset(name="customer_data_asset")
batch_definition = data_asset.add_batch_definition_whole_dataframe("full_batch")

print("Data source : pandas_datasource")
print("Data asset  : customer_data_asset")
print("Batch defn  : full_batch")

Data source : pandas_datasource
Data asset  : customer_data_asset
Batch defn  : full_batch



## Part 2: Create Expectation Suite

Suite name: **`customer_data_expectations`**

| # | Column / Scope | Expectation |
|---|---------------|-------------|
| 1 | `customer_id` | Not null |
| 2 | `customer_id` | Unique |
| 3 | `age` | Between 0 and 120 |
| 4 | `email` | Valid email regex |
| 5 | `salary` | Present in ≥95% of rows |
| 6 | `country` | In {USA, Canada, UK, Australia} |
| 7 | `signup_date` | Datetime type |
| 8 | Table | Row count 500–1000 |


In [116]:
# 2.1  Create the expectation suite
suite = context.suites.add(
    gx.ExpectationSuite(name="customer_data_expectations")
)
print(f"Suite created: {suite.name}")

Suite created: customer_data_expectations


In [117]:
# 2.2  Add all 8 expectations

# Expectation 1 - customer_id not null
suite.add_expectation(
    gxe.ExpectColumnValuesToNotBeNull(column="customer_id")
)
print("Exp 1: customer_id - not null")

# Expectation 2 - customer_id unique
suite.add_expectation(
    gxe.ExpectColumnValuesToBeUnique(column="customer_id")
)
print("Exp 2: customer_id - unique")

# Expectation 3 - age between 0 and 120
suite.add_expectation(
    gxe.ExpectColumnValuesToBeBetween(column="age", min_value=0, max_value=120)
)
print("Exp 3: age - between 0 and 120")

# Expectation 4 - email valid format using regex
EMAIL_REGEX = r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"
suite.add_expectation(
    gxe.ExpectColumnValuesToMatchRegex(column="email", regex=EMAIL_REGEX)
)
print("Exp 4: email - valid regex format")

# Expectation 5 - salary present in at least 95% of rows
suite.add_expectation(
    gxe.ExpectColumnValuesToNotBeNull(column="salary", mostly=0.95)
)
print("Exp 5: salary - >=95% non-null (mostly=0.95)")

# Expectation 6 - country in allowed set
suite.add_expectation(
    gxe.ExpectColumnValuesToBeInSet(
        column="country",
        value_set=["USA", "Canada", "UK", "Australia"]
    )
)
print("Exp 6: country - in {USA, Canada, UK, Australia}")

# Expectation 7 - signup_date is datetime type
suite.add_expectation(
    gxe.ExpectColumnValuesToBeOfType(column="signup_date", type_="datetime64[us]")
)
print("Exp 7: signup_date - datetime type")

# Expectation 8 - row count between 500 and 1000
suite.add_expectation(
    gxe.ExpectTableRowCountToBeBetween(min_value=500, max_value=1000)
)
print("Exp 8: table - row count between 500 and 1000")

print(f"\n Total expectations: {len(suite.expectations)}")

Exp 1: customer_id - not null
Exp 2: customer_id - unique
Exp 3: age - between 0 and 120
Exp 4: email - valid regex format
Exp 5: salary - >=95% non-null (mostly=0.95)
Exp 6: country - in {USA, Canada, UK, Australia}
Exp 7: signup_date - datetime type
Exp 8: table - row count between 500 and 1000

 Total expectations: 8


## Part 3: Data Quality Report

In [118]:
# 3.1  Prepare DataFrame
# Convert signup_date to datetime64 so type expectation validates correctly
df = df_raw.copy()
df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")

print(f"Rows             : {len(df)}")
print(f"signup_date dtype: {df['signup_date'].dtype}")

Rows             : 5015
signup_date dtype: datetime64[us]


In [119]:
# 3.2  Create Validation Definition + Checkpoint, then run
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="customer_validation",
        data=batch_definition,
        suite=suite,
    )
)

checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name="customer_checkpoint",
        validation_definitions=[validation_definition],
        result_format=gx.ResultFormat.COMPLETE,
    )
)

print("Running validation suite...")
checkpoint_result = checkpoint.run(batch_parameters={"dataframe": df})


print(f"  Overall validation success: {checkpoint_result.success}")


Running validation suite...


Calculating Metrics:   0%|          | 0/42 [00:00<?, ?it/s]

  Overall validation success: False


In [120]:
# 3.3  Per-expectation results 
rows = []
for vr in checkpoint_result.run_results.values():
    for r in vr["results"]:
        exp_type  = r["expectation_config"]["type"]
        col       = r["expectation_config"]["kwargs"].get("column", "TABLE")
        passed    = r["success"]
        stats     = r.get("result", {})
        elem_cnt  = stats.get("element_count", "-")
        unexp_cnt = stats.get("unexpected_count", "-")
        unexp_pct = stats.get("unexpected_percent")
        pct_str   = f"{unexp_pct:.1f}%" if unexp_pct is not None else "-"
        rows.append({
            "Column"       : col,
            "Expectation"  : exp_type,
            "Passed"       : "PASS" if passed else "FAIL",
            "Element Count": elem_cnt,
            "Unexpected #" : unexp_cnt,
            "Unexpected %" : pct_str,
        })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

     Column                          Expectation Passed Element Count Unexpected # Unexpected %
customer_id  expect_column_values_to_not_be_null   FAIL          5015          150         3.0%
customer_id    expect_column_values_to_be_unique   FAIL          5015          568        11.7%
        age   expect_column_values_to_be_between   FAIL          5015          384         7.9%
      email  expect_column_values_to_match_regex   FAIL          5015          346         7.6%
     salary  expect_column_values_to_not_be_null   FAIL          5015          425         8.5%
    country    expect_column_values_to_be_in_set   FAIL          5015          301         6.1%
signup_date   expect_column_values_to_be_of_type   PASS             -            -            -
      TABLE expect_table_row_count_to_be_between   FAIL             -            -            -


In [121]:
# 3.4  Full data quality audit - counts per issue
EMAIL_RE = r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"
total = len(df_raw)

issues = {
    "Total rows (expected 500-1000)"  : total,
    "Null customer_id"                : int(df_raw["customer_id"].isna().sum()),
    "Duplicate customer_id"           : int(df_raw.duplicated(subset=["customer_id"]).sum()),
    "Fully duplicate rows"            : int(df_raw.duplicated().sum()),
    "Age out of range (< 0 or > 120)" : int(((df_raw["age"] < 0) | (df_raw["age"] > 120)).sum()),
    "Missing age"                     : int(df_raw["age"].isna().sum()),
    "Invalid email format"            : int(
        df_raw["email"].dropna()
        .apply(lambda x: not bool(re.match(EMAIL_RE, str(x)))).sum()
    ),
    "Missing email"                   : int(df_raw["email"].isna().sum()),
    "Missing salary (fails 95% rule)" : int(df_raw["salary"].isna().sum()),
    "Negative salary"                 : int((df_raw["salary"] < 0).sum()),
    "Invalid country value"           : int(
        df_raw["country"].dropna()
        .apply(lambda x: x not in {"USA","Canada","UK","Australia"}).sum()
    ),
    "Missing country"                 : int(df_raw["country"].isna().sum()),
    "Missing phone"                   : int(df_raw["phone"].isna().sum()),
    "Missing signup_date"             : int(df_raw["signup_date"].isna().sum()),
}


print("    DATA QUALITY ISSUES - FULL AUDIT")
print()
display(pd.DataFrame(list(issues.items()), columns=["Issue", "Count"]))

    DATA QUALITY ISSUES - FULL AUDIT



,Issue,Count
0,Total rows (expected 500-1000),5015
1,Null customer_id,150
2,Duplicate customer_id,452
3,Fully duplicate rows,15
4,Age out of range (< 0 or > 120),384
5,Missing age,147
6,Invalid email format,346
7,Missing email,438
8,Missing salary (fails 95% rule),425
9,Negative salary,159


In [122]:
# 3.5  Generate GE HTML data documentation
context.build_data_docs()

validation_htmls = glob.glob(
    f"{GE_ROOT}/gx/uncommitted/data_docs/local_site/validations/**/*.html",
    recursive=True
)
index_html = f"{GE_ROOT}/gx/uncommitted/data_docs/local_site/index.html"

print(" GE data docs generated:")
print(f"   Folder : {GE_ROOT}/gx/uncommitted/data_docs/local_site/")
print(f"   index.html exists    : {os.path.exists(index_html)}")
if validation_htmls:
    print(f"   validation html      : {validation_htmls[0]}")
print()

 GE data docs generated:
   Folder : gx_project/gx/uncommitted/data_docs/local_site/
   index.html exists    : True
   validation html      : gx_project/gx/uncommitted/data_docs/local_site/validations\customer_data_expectations\__none__\20260626T234754.376915Z\pandas_datasource-customer_data_asset.html



In [123]:
# 3.6  Save standalone HTML validation report
overall = checkpoint_result.success

# Friendly names for expectations (cleaner display in HTML)
FRIENDLY_NAMES = {
    "ExpectColumnValuesToNotBeNull"   : "Must not be null",
    "ExpectColumnValuesToBeUnique"    : "Must be unique",
    "ExpectColumnValuesToBeBetween"   : "Must be between 0 and 120",
    "ExpectColumnValuesToMatchRegex"  : "Must match valid email format (regex)",
    "ExpectColumnValuesToBeInSet"     : "Must be in {USA, Canada, UK, Australia}",
    "ExpectColumnValuesToBeOfType"    : "Must be datetime type",
    "ExpectTableRowCountToBeBetween"  : "Row count must be between 500 and 1000",
}

exp_rows_html = ""
for _, row in results_df.iterrows():
    bg      = "#d4edda" if row["Passed"] == "PASS" else "#f8d7da"
    friendly = FRIENDLY_NAMES.get(row["Expectation"], row["Expectation"])
    exp_rows_html += (
        '<tr style="background:' + bg + '">'
        '<td>' + str(row["Column"]) + '</td>'
        '<td>' + friendly + '</td>'
        '<td style="text-align:center;font-weight:bold">' + str(row["Passed"]) + '</td>'
        '<td style="text-align:right">' + str(row["Element Count"]) + '</td>'
        '<td style="text-align:right">' + str(row["Unexpected #"]) + '</td>'
        '<td style="text-align:right">' + str(row["Unexpected %"]) + '</td>'
        '</tr>\n'
    )

issue_rows_html = ""
for k, v in issues.items():
    issue_rows_html += '<tr><td>' + k + '</td><td style="text-align:right">' + str(v) + '</td></tr>\n'

badge_bg    = "#d4edda" if overall else "#f8d7da"
badge_color = "#155724" if overall else "#721c24"
result_lbl  = "PASSED" if overall else "FAILED (expected - dataset is intentionally messy)"

html_lines = [
    "<!DOCTYPE html><html><head><meta charset='utf-8'>",
    "<title>GE Validation Report</title><style>",
    "body{font-family:Arial,sans-serif;margin:30px;color:#212529}",
    "h1{color:#343a40}h2{color:#495057;border-bottom:2px solid #dee2e6;padding-bottom:4px}",
    "table{border-collapse:collapse;width:100%;margin-bottom:30px;font-size:14px}",
    "th{background:#343a40;color:white;padding:10px;text-align:left}",
    "td{padding:8px 10px;border-bottom:1px solid #dee2e6}",
    ".badge{font-size:15px;font-weight:bold;padding:8px 16px;border-radius:4px;display:inline-block;margin:10px 0}",
    "</style></head><body>",
    "<h1>Great Expectations - Validation Report</h1>",
    f"<p><strong>Suite:</strong> customer_data_expectations &nbsp;|&nbsp; <strong>GE:</strong> {gx.__version__} &nbsp;|&nbsp; <strong>Rows:</strong> {total}</p>",
    f'<div class="badge" style="background:{badge_bg};color:{badge_color}">Overall Result: {result_lbl}</div>',
    "<h2>Expectation Results (8 expectations)</h2><table>",
    "<thead><tr><th>Column</th><th>Expectation</th><th>Passed</th><th>Element Count</th><th>Unexpected #</th><th>Unexpected %</th></tr></thead>",
    "<tbody>" + exp_rows_html + "</tbody></table>",
    "<h2>All Data Quality Issues Found</h2><table>",
    "<thead><tr><th>Issue</th><th>Count</th></tr></thead>",
    "<tbody>" + issue_rows_html + "</tbody></table>",
    "</body></html>",
]

REPORT_HTML = "ge_validation_report.html"
with open(REPORT_HTML, "w") as fh:
    fh.write("\n".join(html_lines))

print(f"HTML report saved: {REPORT_HTML}")

import webbrowser, os
webbrowser.open(os.path.abspath(REPORT_HTML))

HTML report saved: ge_validation_report.html


True

## Part 4: pytest Unit Tests

| Function | Tests | Cases covered |
|----------|-------|---------------|
| `load_csv` | 5 | File not found, empty file, successful load, columns, row count |
| `clean_phone` | 10 | Dashes, dots, parentheses, plain digits, spaces, too few/many digits, letters, None, negative |
| `validate_email` | 14 | Valid formats, invalid formats, edge cases |


In [124]:
# 4.1  Define the three data utility functions

def load_csv(filepath: str) -> pd.DataFrame:
    """
    Load a CSV file into a pandas DataFrame.
    Raises FileNotFoundError if path missing, ValueError if file is empty.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    df_loaded = pd.read_csv(filepath)
    if df_loaded.empty:
        raise ValueError(f"File is empty: {filepath}")
    return df_loaded


def clean_phone(phone) -> str:
    """
    Normalise any phone number to XXX-XXX-XXXX.
    Returns '' for None or inputs that do not yield exactly 10 digits.
    """
    if phone is None:
        return ""
    digits = re.sub(r"\D", "", str(phone))
    if len(digits) != 10:
        return ""
    return f"{digits[:3]}-{digits[3:6]}-{digits[6:]}"


def validate_email(email) -> bool:
    """
    Return True if email matches RFC-5321 compatible format.
    Strips leading/trailing whitespace before matching.
    """
    if not email or not isinstance(email, str):
        return False
    pattern = r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$"
    return bool(re.match(pattern, email.strip()))


print("  Utility functions defined")
print("   load_csv       - loads CSV, raises on missing/empty file")
print("   clean_phone    - normalises any phone format to XXX-XXX-XXXX")
print("   validate_email - returns True/False for email validity")

  Utility functions defined
   load_csv       - loads CSV, raises on missing/empty file
   clean_phone    - normalises any phone format to XXX-XXX-XXXX
   validate_email - returns True/False for email validity


In [125]:
# 4.2  Write test_data_utils.py to disk
TEST_FILE = "test_data_utils.py"

with open(TEST_FILE, "w") as f:
    f.write("\"\"\"\nMAI201 Assignment 2 - pytest Unit Tests\nFunctions: load_csv | clean_phone | validate_email\n\"\"\"\nimport os, re\nimport pandas as pd\nimport pytest\n\n\ndef load_csv(filepath: str) -> pd.DataFrame:\n    if not os.path.exists(filepath):\n        raise FileNotFoundError(f\"File not found: {filepath}\")\n    df = pd.read_csv(filepath)\n    if df.empty:\n        raise ValueError(f\"File is empty: {filepath}\")\n    return df\n\ndef clean_phone(phone) -> str:\n    if phone is None:\n        return \"\"\n    digits = re.sub(r\"\\D\", \"\", str(phone))\n    if len(digits) != 10:\n        return \"\"\n    return f\"{digits[:3]}-{digits[3:6]}-{digits[6:]}\"\n\ndef validate_email(email) -> bool:\n    if not email or not isinstance(email, str):\n        return False\n    pattern = r\"^[a-zA-Z0-9._%+\\-]+@[a-zA-Z0-9.\\-]+\\.[a-zA-Z]{2,}$\"\n    return bool(re.match(pattern, email.strip()))\n\n\nclass TestLoadCsv:\n\n    def test_file_not_found_raises(self):\n        with pytest.raises(FileNotFoundError, match=\"File not found\"):\n            load_csv(\"nonexistent_xyz.csv\")\n\n    def test_empty_file_raises(self, tmp_path):\n        f = tmp_path / \"empty.csv\"\n        f.write_text(\"customer_id,age,email\\n\")\n        with pytest.raises(ValueError, match=\"File is empty\"):\n            load_csv(str(f))\n\n    def test_successful_load_returns_dataframe(self, tmp_path):\n        f = tmp_path / \"good.csv\"\n        f.write_text(\"customer_id,age,email\\nC001,30,test@example.com\\n\")\n        result = load_csv(str(f))\n        assert isinstance(result, pd.DataFrame) and len(result) == 1\n\n    def test_correct_columns(self, tmp_path):\n        f = tmp_path / \"cols.csv\"\n        f.write_text(\"a,b,c\\n1,2,3\\n\")\n        assert list(load_csv(str(f)).columns) == [\"a\", \"b\", \"c\"]\n\n    def test_all_rows_loaded(self, tmp_path):\n        f = tmp_path / \"rows.csv\"\n        f.write_text(\"id,val\\n\" + \"\\n\".join(f\"{i},{i*2}\" for i in range(50)))\n        assert len(load_csv(str(f))) == 50\n\n\nclass TestCleanPhone:\n\n    def test_dashes(self):\n        assert clean_phone(\"416-555-1234\") == \"416-555-1234\"\n\n    def test_dots(self):\n        assert clean_phone(\"416.555.1234\") == \"416-555-1234\"\n\n    def test_parentheses(self):\n        assert clean_phone(\"(416) 555-1234\") == \"416-555-1234\"\n\n    def test_plain_digits(self):\n        assert clean_phone(\"4165551234\") == \"416-555-1234\"\n\n    def test_spaces(self):\n        assert clean_phone(\"416 555 1234\") == \"416-555-1234\"\n\n    def test_too_few_digits(self):\n        assert clean_phone(\"12345\") == \"\"\n\n    def test_too_many_digits(self):\n        assert clean_phone(\"14165551234\") == \"\"\n\n    def test_letters_only(self):\n        assert clean_phone(\"ABCDEFGHIJ\") == \"\"\n\n    def test_none_input(self):\n        assert clean_phone(None) == \"\"\n\n    def test_negative_number(self):\n        assert clean_phone(\"-8437\") == \"\"\n\n\nclass TestValidateEmail:\n\n    def test_standard_email(self):\n        assert validate_email(\"user@example.com\") is True\n\n    def test_subdomain(self):\n        assert validate_email(\"user@mail.example.co.uk\") is True\n\n    def test_plus_tag(self):\n        assert validate_email(\"user+tag@example.com\") is True\n\n    def test_numeric_local(self):\n        assert validate_email(\"1234@example.com\") is True\n\n    def test_uppercase(self):\n        assert validate_email(\"User.Name@Example.COM\") is True\n\n    def test_missing_at(self):\n        assert validate_email(\"userexample.com\") is False\n\n    def test_missing_domain(self):\n        assert validate_email(\"user@\") is False\n\n    def test_missing_tld(self):\n        assert validate_email(\"user@domain\") is False\n\n    def test_at_only(self):\n        assert validate_email(\"@domain.com\") is False\n\n    def test_double_at(self):\n        assert validate_email(\"user@@domain.com\") is False\n\n    def test_empty_string(self):\n        assert validate_email(\"\") is False\n\n    def test_none_input(self):\n        assert validate_email(None) is False\n\n    def test_whitespace_only(self):\n        assert validate_email(\"   \") is False\n\n    def test_surrounding_whitespace(self):\n        assert validate_email(\"  user@example.com  \") is True\n")

print(f" {TEST_FILE} written")
print("   TestLoadCsv      : 5 tests")
print("   TestCleanPhone   : 10 tests")
print("   TestValidateEmail: 14 tests")
print("   Total            : 29 tests")

 test_data_utils.py written
   TestLoadCsv      : 5 tests
   TestCleanPhone   : 10 tests
   TestValidateEmail: 14 tests
   Total            : 29 tests


In [126]:
# 4.3  Run pytest  <-- SCREENSHOT THIS OUTPUT FOR YOUR REPORT
result = subprocess.run(
    [sys.executable, "-m", "pytest", TEST_FILE, "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode == 0:
    print("\n All 29 tests PASSED")
else:
    print("\n Some tests FAILED")
    print(result.stderr)

============================= test session starts =============================
collecting ... collected 29 items

test_data_utils.py::TestLoadCsv::test_file_not_found_raises PASSED       [  3%]
test_data_utils.py::TestLoadCsv::test_empty_file_raises PASSED           [  6%]
test_data_utils.py::TestLoadCsv::test_successful_load_returns_dataframe PASSED [ 10%]
test_data_utils.py::TestLoadCsv::test_correct_columns PASSED             [ 13%]
test_data_utils.py::TestLoadCsv::test_all_rows_loaded PASSED             [ 17%]
test_data_utils.py::TestCleanPhone::test_dashes PASSED                   [ 20%]
test_data_utils.py::TestCleanPhone::test_dots PASSED                     [ 24%]
test_data_utils.py::TestCleanPhone::test_parentheses PASSED              [ 27%]
test_data_utils.py::TestCleanPhone::test_plain_digits PASSED             [ 31%]
test_data_utils.py::TestCleanPhone::test_spaces PASSED                   [ 34%]
test_data_utils.py::TestCleanPhone::test_too_few_digits PASSED           [ 37%]

## Part 5: Reflection

### Data Quality Issues Found

| Issue | Count |
|-------|------:|
| Total rows (expected 500–1000) | **5,015** FAIL |
| Null `customer_id` | 150 |
| Duplicate `customer_id` | 452 |
| Fully duplicate rows | 15 |
| Age out of range (< 0 or > 120) | 384 |
| Missing age | 147 |
| Invalid email format | 346 |
| Missing email | 438 |
| Missing salary (fails 95% rule) | 425 |
| Negative salary | 159 |
| Invalid country value | 301 |
| Missing country | 41 |
| Missing phone | 319 |
| Missing signup_date | 14 |

### Which issue would most impact ML model performance?

**Missing and invalid `salary` values** would cause the greatest harm. Salary is a high-signal continuous feature in customer behaviour models. With ~8.5% missing and ~3.2% negative, nearly 12% of records are absent or nonsensical. Mean/median imputation introduces systematic bias when missingness is not at random — for instance if high earners disproportionately omit salary. Negative values distort normalisation statistics and can destabilise gradient-based optimisers. By contrast, phone format inconsistencies carry no predictive signal, and fully duplicate rows can be dropped in pre-processing without distorting the learned distribution.
